# Silver Layer: Data Quality and Standardization

This notebook reads the Bronze Parquet datasets, applies lightweight type normalization and duplicate handling, and writes the cleaned datasets to `Silver/silver`. It also runs focused checks for nulls, invalid dates, negative prices, invalid payments, and review scores outside the expected 1-5 range.

## Inputs
- Parquet datasets under `../Bronze/staging`
- A local PySpark installation

## Transformations
- Cast identifiers and financial fields to usable numeric types.
- Parse order and review date fields as timestamps.
- Remove duplicate customers and orders using their business identifiers.
- Profile row counts, distinct rows, and null counts.

## Outputs
Cleaned Parquet datasets are written under `Silver/silver` for the Gold dimensional model. Review the validation output before promoting data downstream.

In [1]:
from pyspark.sql import SparkSession

In [3]:
spark = SparkSession.builder.appName("Building Silver Medallion Workflow").getOrCreate()
spark

c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [5]:
customers_df = spark.read.parquet("../Bronze/staging/customers")
category_df = spark.read.parquet("../Bronze/staging/category")
orders_df = spark.read.parquet("../Bronze/staging/orders")
order_items_df = spark.read.parquet("../Bronze/staging/order_items")
order_payments_df = spark.read.parquet("../Bronze/staging/order_payments")
sellers_df = spark.read.parquet("../Bronze/staging/sellers")
order_reviews_df = spark.read.parquet("../Bronze/staging/order_reviews")
geolocation_df = spark.read.parquet("../Bronze/staging/geolocation")
products_df = spark.read.parquet("../Bronze/staging/products")

In [6]:
for name, df in [
    ("customers", customers_df), ("category", category_df), ("orders", orders_df),
    ("order_items", order_items_df), ("order_payments", order_payments_df),
    ("sellers", sellers_df), ("order_reviews", order_reviews_df),
    ("geolocation", geolocation_df), ("products", products_df)
]:
    print(f"{name}: {df.count()} rows")

customers: 99441 rows
category: 71 rows
orders: 99441 rows
order_items: 112650 rows
order_payments: 103886 rows
sellers: 3095 rows
order_reviews: 99224 rows
geolocation: 1000163 rows
products: 32951 rows


In [7]:
from pyspark.sql import functions as F

def profile_table(df, name):
    print(f"\n--- {name} ---")
    print(f"Row count: {df.count()}")
    print(f"Distinct rows: {df.distinct().count()}")
    df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

In [8]:
customers_silver = customers_df \
    .withColumn("customer_zip_code_prefix", F.col("customer_zip_code_prefix").cast("integer")) \
    .dropDuplicates(["customer_id"])

profile_table(customers_silver, "customers_silver")


--- customers_silver ---
Row count: 99441
Distinct rows: 99441
+-----------+------------------+------------------------+-------------+--------------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|
+-----------+------------------+------------------------+-------------+--------------+
|          0|                 0|                       0|            0|             0|
+-----------+------------------+------------------------+-------------+--------------+



In [9]:
orders_silver = orders_df \
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date")) \
    .dropDuplicates(["order_id"])

# sanity check: delivery shouldn't happen before purchase
bad_dates = orders_silver.filter(
    F.col("order_delivered_customer_date") < F.col("order_purchase_timestamp")
)
print(f"Orders with impossible delivery dates: {bad_dates.count()}")

Orders with impossible delivery dates: 0


In [10]:
order_items_silver = order_items_df \
    .withColumn("order_item_id", F.col("order_item_id").cast("integer")) \
    .withColumn("price", F.col("price").cast("double")) \
    .withColumn("freight_value", F.col("freight_value").cast("double")) \
    .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))

print(f"Negative prices: {order_items_silver.filter(F.col('price') < 0).count()}")

Negative prices: 0


In [12]:
order_payments_silver = order_payments_df \
    .withColumn("payment_sequential", F.col("payment_sequential").cast("integer")) \
    .withColumn("payment_installments", F.col("payment_installments").cast("integer")) \
    .withColumn("payment_value", F.col("payment_value").cast("double"))

print(f"Negative/zero payments: {order_payments_silver.filter(F.col('payment_value') <= 0).count()}")

Negative/zero payments: 9


In [14]:
order_reviews_silver = order_reviews_df \
    .withColumn("review_score", F.col("review_score").cast("integer")) \
    .withColumn("review_creation_date", F.to_timestamp("review_creation_date")) \
    .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp"))

print(f"Reviews outside 1-5 range: {order_reviews_silver.filter((F.col('review_score') < 1) | (F.col('review_score') > 5)).count()}")

Reviews outside 1-5 range: 0


In [15]:
def write_silver(df, table_name):
    df.write.mode("overwrite").parquet(f"silver/{table_name}")
    print(f"silver/{table_name} written")


write_silver(customers_df, "customers")
write_silver(category_df, ".category")
write_silver(orders_df, "orders")
write_silver(order_items_df, "order_items")
write_silver(order_payments_df, "order_payments")
write_silver(sellers_df, "sellers")
write_silver(order_reviews_df, "order_reviews")
write_silver(geolocation_df, "geolocation")
write_silver(products_df, "products")

silver/customers written
silver/.category written
silver/orders written
silver/order_items written
silver/order_payments written
silver/sellers written
silver/order_reviews written
silver/geolocation written
silver/products written
